# Beta, stratification, and Burger-number hypotheses

This notebook asks whether the equatorward increase in measured `TiltDis` is better described by beta alone, stratification, `N2/f²`, a transparent deformation-radius/Burger proxy, or background shear. Beta is inseparable from latitude, so spatial robustness and comparisons among competing predictors are central.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


In [ ]:
N2_CACHE = Path("/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day_v2.parquet")
n2 = pd.read_parquet(N2_CACHE)
data = mech.merge_one_to_one_or_many_to_one(df, n2)
data = tilt.add_pv_gradient_terms(data, grid)
data = mech.add_stratification_proxies(data)
data = mech.add_tilt_components(data)
data[["TiltDis", "beta", "N2_500m_s2", "N2_over_f2_500m", "Bu_proxy_500m"]].describe()


In [ ]:
predictors = ["beta", "N2_200m_s2", "N2_500m_s2", "N2_over_f2_200m",
              "N2_over_f2_500m", "Rd_proxy_500m_km", "Bu_proxy_500m", "Rc", "h"]
eddy = data.groupby(["Cyc", "Eddy"], as_index=False)[["TiltDis", *predictors]].median()
rows = []
from scipy.stats import spearmanr
for cyc, part in eddy.groupby("Cyc"):
    for predictor in predictors:
        use = part[[predictor, "TiltDis"]].dropna()
        rho, p = spearmanr(use[predictor], use["TiltDis"])
        rows.append({"Cyc": cyc, "predictor": predictor, "eddies": len(use), "rho": rho, "p": p})
display(pd.DataFrame(rows).sort_values(["Cyc", "rho"], ascending=[True, False]))


In [ ]:
# Compare predeclared clustered models; coefficients are standardised for scale comparability.
import statsmodels.formula.api as smf

model_sets = {
    "structure_environment": ["Rc", "h"],
    "plus_beta": ["Rc", "h", "beta"],
    "plus_N2": ["Rc", "h", "beta", "N2_500m_s2"],
    "N2_over_f2": ["Rc", "h", "N2_over_f2_500m"],
    "Burger_proxy": ["Rc", "h", "Bu_proxy_500m"],
}
fits, comparison = {}, []
for cyc, part in data.groupby("Cyc"):
    for name, cols in model_sets.items():
        use = part[["Eddy", "TiltDis", *cols]].dropna().copy()
        for col in cols:
            use[f"z_{col}"] = (use[col] - use[col].mean()) / use[col].std()
        fit = smf.gee("np.log1p(TiltDis) ~ " + " + ".join(f"z_{c}" for c in cols),
                      groups="Eddy", data=use).fit()
        fits[(cyc, name)] = fit
        qic = fit.qic()[0] if callable(getattr(fit, "qic", None)) else np.nan
        comparison.append({"Cyc": cyc, "model": name, "rows": len(use),
                           "eddies": use.Eddy.nunique(), "QIC": qic})
display(pd.DataFrame(comparison))


In [ ]:
# Spatial sensitivity: repeat within broad latitude and shelf regimes.
data["latitude_band"] = pd.cut(data["lat" if "lat" in data else "yc"], 4)
spatial = (data.groupby(["Cyc", "latitude_band"], observed=True)
           .agg(rows=("TiltDis", "size"), eddies=("Eddy", "nunique"),
                tilt=("TiltDis", "median"), n2=("N2_500m_s2", "median"),
                beta=("beta", "median"), burger=("Bu_proxy_500m", "median")))
display(spatial)


## Interpretation guardrail

Call the equatorward pattern specifically beta-related only if beta adds information beyond structure, bathymetry, stratification and shear and survives spatial blocking. `Rd_proxy` and `Bu_proxy` are constant-N screening proxies, not solved vertical modes; label them as such in figures and text.
